# Análisis Exploratorio de Datos - Hubway Bike Sharing

## Autor
Guerra Chura Joan Leonardo

## Objetivo
Realizar un EDA completo del sistema Hubway incorporando Data Wrangling, análisis estadístico, visualizaciones temporales, espaciales y enriquecimiento con datos climáticos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",50)


# Carga de datos

In [ ]:
stations = pd.read_csv("../data/hubway_stations.csv")
trips = pd.read_csv("../data/hubway_trips.csv", low_memory=False)

print(stations.shape)
print(trips.shape)


# Paso 0: Metadata

El dataset contiene estaciones y viajes.

- stations: dimensión espacial.
- trips: hechos transaccionales de movilidad.

Cada fila de trips representa un viaje individual.

In [ ]:
trips.head()


# Paso 1: Data Wrangling

Se revisa calidad de datos, tipos, duplicados y valores faltantes.

In [ ]:
trips['start_dt']=pd.to_datetime(trips['start_date'])
trips['end_dt']=pd.to_datetime(trips['end_date'])

trips.info()


In [ ]:
missing=(trips.isna().sum()/len(trips)*100).sort_values(ascending=False)

plt.figure(figsize=(9,4))
missing[missing>0].plot(kind='bar')
plt.title("Porcentaje de valores faltantes")
plt.ylabel("%")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.heatmap(trips.isna().sample(500), cbar=False)
plt.title("Mapa de valores faltantes")
plt.show()


In [ ]:
pd.crosstab(trips['subsc_type'],trips['gender'].isna(),normalize='index')


# Paso 2: Outliers

In [ ]:
q1=trips.duration.quantile(.25)
q3=trips.duration.quantile(.75)
iqr=q3-q1
upper=q3+1.5*iqr

print("Límite superior:",upper)
print("Negativos:",(trips.duration<0).sum())
print(">24h:",(trips.duration>86400).sum())


In [ ]:
plt.figure(figsize=(9,4))
sns.histplot(trips.loc[(trips.duration>0)&(trips.duration<5000),'duration'],bins=60)
plt.title("Distribución de duración de viajes")
plt.xlabel("Segundos")
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=trips[trips.duration<86400],
            x='subsc_type',
            y='duration')
plt.title("Duración por tipo de usuario")
plt.show()


# Paso 3: Visualización avanzada

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=trips,x='subsc_type')
plt.title("Registered vs Casual")
plt.show()


In [ ]:
trips['hour']=trips.start_dt.dt.hour

plt.figure(figsize=(9,4))
trips.hour.value_counts().sort_index().plot(kind='bar')
plt.title("Viajes por hora del día")
plt.xlabel("Hora")
plt.ylabel("Viajes")
plt.show()


In [ ]:
trips['weekday']=trips.start_dt.dt.day_name()

heat=trips.pivot_table(index='weekday',
                        columns='hour',
                        values='hubway_id',
                        aggfunc='count')

plt.figure(figsize=(12,5))
sns.heatmap(heat,cmap='viridis')
plt.title("Heatmap hora vs día")
plt.show()


In [ ]:
monthly=trips.groupby(trips.start_dt.dt.to_period('M')).size()

plt.figure(figsize=(12,4))
monthly.plot(marker='o')
plt.title("Estacionalidad mensual")
plt.ylabel("Viajes")
plt.show()


In [ ]:
top_start=trips.strt_statn.value_counts().head(10)

plt.figure(figsize=(8,4))
top_start.plot(kind='bar')
plt.title("Top estaciones de salida")
plt.ylabel("Viajes")
plt.show()


In [ ]:
top_end=trips.end_statn.value_counts().head(10)

plt.figure(figsize=(8,4))
top_end.plot(kind='bar')
plt.title("Top estaciones de llegada")
plt.ylabel("Viajes")
plt.show()


In [ ]:
usage=trips.strt_statn.value_counts()

mapa=stations.merge(usage.rename('demanda'),
                    left_on='id',
                    right_index=True,
                    how='left')

plt.figure(figsize=(8,6))
plt.scatter(mapa.lng,
            mapa.lat,
            s=mapa.demanda.fillna(0)/50,
            alpha=.5)

plt.title("Distribución espacial de demanda")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.show()


# Paso 4: Problema potencial - clima y demanda

Se incorpora Open-Meteo para analizar si temperatura y precipitación afectan la demanda.

In [ ]:
trips['fecha']=trips.start_dt.dt.date

daily=trips.groupby('fecha').size().reset_index(name='viajes')

params={
'latitude':42.355,
'longitude':-71.065,
'start_date':'2011-07-28',
'end_date':'2013-11-30',
'daily':'temperature_2m_max,precipitation_sum,snowfall_sum',
'timezone':'America/New_York'
}

weather=requests.get(
'https://archive-api.open-meteo.com/v1/archive',
params=params
).json()

clima=pd.DataFrame(weather['daily'])
clima['time']=pd.to_datetime(clima['time']).dt.date

datos=daily.merge(clima,left_on='fecha',right_on='time')
datos.head()


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))

sns.regplot(data=datos,
            x='temperature_2m_max',
            y='viajes',
            ax=ax[0])

ax[0].set_title("Temperatura vs viajes")

sns.regplot(data=datos,
            x='precipitation_sum',
            y='viajes',
            ax=ax[1])

ax[1].set_title("Precipitación vs viajes")

plt.show()


In [ ]:
print("Correlación temperatura:",
datos['viajes'].corr(datos['temperature_2m_max']))

print("Correlación lluvia:",
datos['viajes'].corr(datos['precipitation_sum']))


# Conclusiones

El análisis evidencia patrones temporales, espaciales y ambientales.

Los principales hallazgos son:

- existencia de patrones de movilidad por hora y día;
- diferencias entre usuarios registrados y casuales;
- concentración espacial de demanda;
- presencia de problemas de calidad de datos;
- influencia potencial del clima sobre la utilización del sistema.

Este análisis permite construir modelos posteriores de predicción de demanda.